# Knowledge Distillation on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CS-7150-DL/Distillation/blob/main/scripts/run_colab.ipynb)

This notebook runs knowledge distillation experiments on Google Colab.

**Setup (one-time):** Add tokens to Colab Secrets (🔑 in left sidebar):
1. `GITHUB_TOKEN`: https://github.com/settings/tokens → Generate new token (classic) → Select `repo` scope
2. `HF_TOKEN`: https://huggingface.co/settings/tokens → Create new token (only needed for Llama models)
3. `WANDB_API_KEY`: https://wandb.ai/authorize (optional, for experiment tracking)

**Hardware:** A100 GPU recommended (T4/V100 also work)

**Quick Configs (~30-60 min):**
- BERT → DistilBERT on MNLI
- BERT → DistilBERT on QQP
- Llama 3B → 1B on translation

**Full Configs (~2-8 hours):**
- Use the non-quick versions for paper reproduction

## 1. Setup Environment

In [1]:
# Mount Google Drive (saves checkpoints even if runtime disconnects)
import os

IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in dir() else False
DRIVE_MOUNTED = False

if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE_MOUNTED = True
        OUTPUT_BASE = "/content/drive/MyDrive/distillation"
        print("Google Drive mounted successfully")
    except Exception as e:
        print(f"Drive mount failed (expected in VS Code + Colab runtime): {e}")
        print("Falling back to local outputs")
        DRIVE_MOUNTED = False
        OUTPUT_BASE = "./outputs"
        os.makedirs(OUTPUT_BASE, exist_ok=True)
else:
    print("Running locally - skipping drive mount")
    OUTPUT_BASE = "../outputs"
    os.makedirs(OUTPUT_BASE, exist_ok=True)

print(f"Output directory: {OUTPUT_BASE}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted successfully
Output directory: /content/drive/MyDrive/distillation


In [2]:
# Keep-alive script (only needed in Colab - prevents disconnection due to inactivity)
if IN_COLAB:
    from IPython.display import Javascript
    display(Javascript('''
        setInterval(() => {
            console.log("Keep alive: " + new Date());
            document.querySelector("colab-connect-button")?.click();
        }, 60000);
    '''))
    print("Keep-alive script running (check browser console)")
else:
    print("Running locally - keep-alive not needed")

<IPython.core.display.Javascript object>

Keep-alive script running (check browser console)


In [3]:
# Clone repository (only in Colab)
if IN_COLAB:
    import os
    if os.path.exists('/content/Distillation'):
        print("Repository already exists, pulling latest changes...")
        %cd /content/Distillation
        !git pull
    else:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
        !git clone https://{token}@github.com/CS-7150-DL/Distillation.git /content/Distillation
        %cd /content/Distillation
else:
    print("Running locally - using current repository")

Repository already exists, pulling latest changes...
/content/Distillation
Already up to date.


In [4]:
# Install dependencies (always run after clone/pull)
if IN_COLAB:
    !pip install -q uv
    !uv pip install -q -e . --system
    print("Dependencies installed successfully")
else:
    print("Running locally - assuming dependencies are already installed")
    print("If not, run: pip install -e . from the project root")

Dependencies installed successfully


## 2. Verify GPU

In [5]:
!nvidia-smi

Wed Jan 28 20:55:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   29C    P0             42W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [6]:
# Verify environment and device
import os
import torch

def detect_environment():
    if os.environ.get("SLURM_JOB_ID"):
        return "slurm"
    elif os.environ.get("COLAB_GPU"):
        return "colab"
    return "local"

def get_device():
    if torch.cuda.is_available():
        return "cuda"
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"
    return "cpu"

print(f"Environment: {detect_environment()}")
print(f"Device: {get_device()}")

Environment: colab
Device: cuda


## 3. Login to Services

In [7]:
# W&B login (for experiment tracking)
# Uses WANDB_API_KEY from Colab Secrets, or disables W&B if unavailable/invalid
import os

WANDB_ENABLED = False

if IN_COLAB:
    try:
        from google.colab import userdata
        api_key = userdata.get('WANDB_API_KEY')
        if api_key:
            os.environ['WANDB_API_KEY'] = api_key
            # Verify the key works by attempting login
            import wandb
            wandb.login(key=api_key, verify=True)
            # Test with a quick API call
            api = wandb.Api()
            _ = api.viewer  # This will raise if permissions are wrong
            WANDB_ENABLED = True
            print("✓ W&B login successful and verified")
        else:
            raise ValueError("WANDB_API_KEY is empty")
    except Exception as e:
        print(f"⚠ W&B setup failed: {e}")
        print("→ Disabling W&B tracking (training will still work)")
        os.environ['WANDB_MODE'] = 'disabled'
        WANDB_ENABLED = False
else:
    # Local mode - check if WANDB_API_KEY is already set
    if os.environ.get('WANDB_API_KEY'):
        try:
            import wandb
            wandb.login(verify=True)
            WANDB_ENABLED = True
            print("✓ W&B login successful (using existing API key)")
        except:
            os.environ['WANDB_MODE'] = 'disabled'
            print("⚠ W&B disabled (invalid API key)")
    else:
        os.environ['WANDB_MODE'] = 'disabled'
        print("ℹ W&B disabled (no API key set)")

print(f"\nW&B tracking: {'enabled' if WANDB_ENABLED else 'disabled'}")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: karthik-ravi-1537 (karthik-ravi-1537-personal) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✓ W&B login successful and verified

W&B tracking: enabled


In [8]:
# HuggingFace login (only needed for Llama models)
# Skip this cell if running BERT experiments
if IN_COLAB:
    try:
        from google.colab import userdata
        from huggingface_hub import login
        login(token=userdata.get('HF_TOKEN'))
        print("HuggingFace login successful")
    except Exception as e:
        print(f"HF login failed: {e}")
        print("Add HF_TOKEN to Colab Secrets for Llama models")

HuggingFace login successful


---
## 4. Run Training

Choose ONE of the following experiments to run.

### Option A: BERT → DistilBERT on MNLI (Recommended Start)

| Version | Time | Expected Accuracy |
|---------|------|-------------------|
| Quick | ~30-45 min | ~0.75-0.78 |
| Full | ~2-3 hours | ~0.817 |

In [9]:
# MNLI - Quick version (~30-45 min)
!python train.py \
    --config configs/bert_distilbert_mnli_quick.yaml \
    --output_dir {OUTPUT_BASE}/mnli_quick

Streaming output truncated to the last 5000 lines.
 91% 140/154 [00:14<00:01,  9.92it/s]
                                       
{'loss_ce': 0.9899215698242188, 'loss_kl': 0.06181930750608444, 'loss_total': 0.5258704423904419, 'epoch': 0.49}
 49% 3000/6136 [06:54<06:16,  8.33it/s]
 92% 141/154 [00:14<00:01,  9.93it/s]
                                       
{'loss_ce': 0.9919281005859375, 'loss_kl': 0.08984345197677612, 'loss_total': 0.5408858060836792, 'epoch': 0.49}
 49% 3000/6136 [06:54<06:16,  8.33it/s]
 92% 142/154 [00:14<00:01,  9.93it/s]
                                       
{'loss_ce': 1.0418014526367188, 'loss_kl': 0.04682313650846481, 'loss_total': 0.5443122982978821, 'epoch': 0.49}
 49% 3000/6136 [06:54<06:16,  8.33it/s]
 93% 143/154 [00:14<00:01,  9.93it/s]
                                       
{'loss_ce': 0.9641876220703125, 'loss_kl': 0.0649099051952362, 'loss_total': 0.5145487785339355, 'epoch': 0.49}
 49% 3000/6136 [06:54<06:16,  8.33it/s]
 94% 144/154 [00:14<00:01,

In [ ]:
# MNLI - Full version (~2-3 hours)
!python train.py \
    --config configs/bert_distilbert_mnli.yaml \
    --output_dir {OUTPUT_BASE}/mnli_full

### Option B: BERT → DistilBERT on QQP

| Version | Time | Expected Accuracy |
|---------|------|-------------------|
| Quick | ~30-45 min | ~0.85-0.88 |
| Full | ~2-3 hours | ~0.900 |

In [ ]:
# QQP - Quick version (~30-45 min)
!python train.py \
    --config configs/bert_distilbert_qqp_quick.yaml \
    --output_dir {OUTPUT_BASE}/qqp_quick

In [ ]:
# QQP - Full version (~2-3 hours)
!python train.py \
    --config configs/bert_distilbert_qqp.yaml \
    --output_dir {OUTPUT_BASE}/qqp_full

### Option C: Llama 3B → 1B Translation (Requires HF Login)

| Version | Time | Expected BLEU |
|---------|------|---------------|
| Quick | ~45-60 min | ~15-20 |
| Full | ~6-8 hours | ~27 |

In [ ]:
# Translation - Quick version (~45-60 min)
!python train.py \
    --config configs/llama_3b_1b_translation_quick.yaml \
    --output_dir {OUTPUT_BASE}/translation_quick

In [ ]:
# Translation - Full version (~6-8 hours)
!python train.py \
    --config configs/llama_3b_1b_translation.yaml \
    --output_dir {OUTPUT_BASE}/translation_full

---
## 5. View Results

In [10]:
# View final metrics
import json
from pathlib import Path

# Uses OUTPUT_BASE from earlier cell
output_dir = f"{OUTPUT_BASE}/mnli_quick"

trainer_state = Path(output_dir) / "trainer_state.json"

if trainer_state.exists():
    with open(trainer_state) as f:
        state = json.load(f)

    print("=" * 50)
    print("TRAINING RESULTS")
    print("=" * 50)

    if 'log_history' in state:
        # Get final eval metrics
        eval_logs = [log for log in state['log_history'] if 'eval_loss' in log]
        if eval_logs:
            final_eval = eval_logs[-1]
            print("\nFinal Evaluation Metrics:")
            for key, value in final_eval.items():
                if key.startswith('eval_'):
                    print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")
else:
    print(f"No trainer state found at {trainer_state}")
    print("Training may still be in progress or output_dir is incorrect.")

No trainer state found at /content/drive/MyDrive/distillation/mnli_quick/trainer_state.json
Training may still be in progress or output_dir is incorrect.


In [11]:
# List saved checkpoints
!ls -la {OUTPUT_BASE}/mnli_quick/

total 262498
drwx------ 2 root root      4096 Jan 28 21:10 checkpoint-6000
drwx------ 2 root root      4096 Jan 28 21:10 checkpoint-6136
-rw------- 1 root root       717 Jan 28 21:10 config.json
-rw------- 1 root root 267835644 Jan 28 21:10 model.safetensors
-rw------- 1 root root       125 Jan 28 21:10 special_tokens_map.json
-rw------- 1 root root      1221 Jan 28 21:10 tokenizer_config.json
-rw------- 1 root root    711661 Jan 28 21:10 tokenizer.json
-rw------- 1 root root      5905 Jan 28 21:10 training_args.bin
-rw------- 1 root root       242 Jan 28 21:10 training_state.json
-rw------- 1 root root    231508 Jan 28 21:10 vocab.txt


---
## 6. Download Model (Optional)

In [12]:
# Download trained model (only needed in Colab with drive mounted)
if IN_COLAB and DRIVE_MOUNTED:
    from google.colab import files
    import shutil

    output_dir = f"{OUTPUT_BASE}/mnli_quick"
    shutil.make_archive("/content/trained_model", 'zip', output_dir)
    files.download("/content/trained_model.zip")
else:
    print(f"Model saved at: {OUTPUT_BASE}/mnli_quick")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## Troubleshooting

**Colab disconnects:**
- Make sure keep-alive cell ran
- On your laptop: run `caffeinate -dims` in terminal
- Checkpoints saved to Google Drive will persist

**Out of memory:**
- Restart runtime and try again
- Use quick configs (smaller batch sizes)

**W&B issues:**
- Set `WANDB_MODE=disabled` to skip W&B:
  ```python
  import os
  os.environ['WANDB_MODE'] = 'disabled'
  ```